# Civil Unrest Event Extraction & Forecasting Pipeline
This notebook demonstrates the end-to-end research pipeline:
1. **Data Cleaning & Preprocessing**: Deduplication, text normalization, and repost discounting.
2. **NLP Event Extraction**: Contextual disambiguation of 'march' triggers and argument extraction.
3. **Spatiotemporal Aggregation**: Causal lag feature construction and ACLED ground-truth alignment.
4. **Model Evaluation**: Multi-class forecasting metrics, confusion matrix, and feature importances.

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set root path
sys.path.insert(0, os.path.abspath(".."))
from src.cleaning import clean_single_post, RawPost
from src.argument_extraction import extract_all_arguments
from src.forecasting import LightGBMForecaster, FEATURE_COLUMNS
from src.evaluation import evaluate_forecasting

## 1. Data Cleaning & Deduplication
We clean raw microblog text, remove near-duplicates via cryptographic hashing, and normalize timestamps.

In [2]:
sample_raw = RawPost(
    tweet_id="1001",
    text="RT @farmer_voice: Massive tractor march scheduled for Singhu border tomorrow 10am! https://t.co/xyz",
    created_at_utc="2021-01-25T10:00:00Z",
    user_id="u_123",
    user_location="Delhi, India"
)

cleaned = clean_single_post(sample_raw)
print(f"Original Text : {sample_raw.text}")
print(f"Cleaned Text  : {cleaned.cleaned_text}")
print(f"Is Repost     : {cleaned.is_repost} (Discounted to avoid crowd inflation)")

## 2. NLP Event Extraction & 'March' Disambiguation
We disambiguate genuine civil unrest mobilizations from calendar months and metaphors, then extract semantic arguments.

In [3]:
test_sentences = [
    "Thousands of farmers announce peaceful tractor march towards Parliament tomorrow morning.",
    "The financial budget session was held in late March 2021 as economic reforms continue.",
    "Nothing can stop the steady march of technology and democracy."
]

for sent in test_sentences:
    args = extract_all_arguments(text=sent, is_march_predicted=True)
    print(f"
Input: {sent}")
    print(f"  -> Trigger  : {args.trigger_text}")
    print(f"  -> Location : {args.location}")
    print(f"  -> Scale    : {args.scale_category}")
    print(f"  -> Status   : {args.event_status}")

## 3. Spatiotemporal Dataset & ACLED Ground Truth
We inspect the aggregated district-day sequences mapped to ACLED conflict ground-truth classes:
- : No Unrest
- : Peaceful Demonstration
- : Violent Escalation / Clash

In [4]:
df_train = pd.read_csv("../data/processed/train.csv")
df_test = pd.read_csv("../data/processed/test.csv")

print(f"Training samples : {len(df_train)}")
print(f"Testing samples  : {len(df_test)}")
print("
Class Distribution (Severe Imbalance):")
print(df_train["label"].value_counts(normalize=True))

## 4. Model Training & Evaluation
We train the LightGBM forecaster using causal lag and engagement velocity features, then evaluate on test data.

In [5]:
model = LightGBMForecaster()
model.fit(df_train, label_col="label")

preds = model.predict(df_test)
probs = model.predict_proba(df_test)

metrics = evaluate_forecasting(y_true=df_test["label"].values, y_pred=preds, y_prob=probs)
print(f"Test Accuracy : {metrics['accuracy']:.4f}")
print(f"Macro F1      : {metrics['macro_f1']:.4f}")
print(f"PR-AUC        : {metrics['pr_auc']:.4f}")

## 5. Confusion Matrix & Feature Importance
Visualizing model performance across unrest classes and the top predictive features.

In [6]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(df_test["label"], preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["0: None", "1: Peaceful", "2: Violent"],
            yticklabels=["0: None", "1: Peaceful", "2: Violent"])
plt.title("24-Hour Regional Forecast Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True ACLED Label")
plt.tight_layout()
plt.show()

# Top Features
top_feats = model.get_top_features(10)
feat_df = pd.DataFrame(top_feats, columns=["Feature", "Importance"])
plt.figure(figsize=(8, 4))
sns.barplot(data=feat_df, x="Importance", y="Feature", palette="viridis")
plt.title("Top 10 Predictive Features (Engagement Velocity & Lags)")
plt.tight_layout()
plt.show()